<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week05-investigation-agents/Nugget024_Investigation_Workflow_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-genai

In [106]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [69]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [110]:
import json

def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  resJsonObj  = responseObj[objname]
  return resJsonObj



In [88]:
investigation = {
    "sla_compliance": 72,
    "approval_latency": 12,
    "inactive_approvers": 45,
    "campaign_size_growth": 5,
    "provisioning_latency": 2
}

In [89]:
hypotheses = []

if investigation["inactive_approvers"] > 20:
    hypotheses.append("Inactive Approvers")

if investigation["approval_latency"] > 8:
    hypotheses.append("Workflow Bottleneck")

if investigation["campaign_size_growth"] > 20:
    hypotheses.append("Campaign Growth")

In [90]:
print(hypotheses)

['Inactive Approvers', 'Workflow Bottleneck']


In [97]:
scores = {}

for h in hypotheses:
    scores[h] = 0

if "Inactive Approvers" in scores:
    scores["Inactive Approvers"] += 3

if "Workflow Bottleneck" in scores:
    scores["Workflow Bottleneck"] += 2

In [92]:
ranked = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True
)

In [93]:
prompt = f"""
You are an IAM Investigation Agent.

Investigation Data:

{investigation}

Hypothesis Ranking:

{ranked}

Generate:

1. Executive Summary

2. Most Likely Causes

3. Confidence Level

4. Recommended Actions

Keep report concise.
"""

In [107]:
print(callGPT(prompt).text)

**IAM Investigation Report**

**1. Executive Summary**
The IAM system is experiencing significant delays in approval processes, evidenced by an approval latency of 12 units and a sub-optimal SLA compliance of 72%. A substantial number of inactive approvers (45%) are a primary contributor to these bottlenecks, hindering efficient access provisioning.

**2. Most Likely Causes**
*   **Inactive Approvers:** The high percentage (45%) of inactive approvers directly leads to stalled approval requests, significantly increasing `approval_latency` and impacting overall `sla_compliance`.
*   **Workflow Bottleneck:** The elevated `approval_latency` (12) points to inefficiencies within the approval workflow, largely exacerbated by inactive approvers who create blockages.

**3. Confidence Level**
High. The investigation data strongly supports the primary hypotheses, showing a direct correlation between inactive approvers and approval latency, impacting SLA compliance.

**4. Recommended Actions**
1. 

In [113]:
prompt = f"""
Investigation Data:

{investigation}

Generate possible causes. Only give causes and keep causes 2 words as per below example
Inactive Approvers
Workflow Bottleneck
Campaign Growth

Return JSON:

{{
  "hypotheses":[]
}}
"""

Generate Hypothesis from LLM

In [114]:
hypotheses = return_json(cleanse_response(callGPT(prompt)),"hypotheses")
print(hypotheses)

['Inactive Approvers', 'Workflow Bottleneck', 'System Delays', 'Process Inefficiency', 'Campaign Growth', 'Training Gaps']


In [115]:
scores = {}

for h in hypotheses:
    scores[h] = 0

if "Inactive Approvers" in scores:
    scores["Inactive Approvers"] += 3

if "Workflow Bottleneck" in scores:
    scores["Workflow Bottleneck"] += 2

if "System Delays" in scores:
    scores["System Delays"] += 1

if "Process Inefficiency" in scores:
    scores["Process Inefficiency"] += 3

if "Campaign Growth" in scores:
    scores["Campaign Growth"] += 4

In [116]:
ranked = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True
)

In [117]:
prompt = f"""
You are an IAM Investigation Agent.

Investigation Data:

{investigation}

Hypothesis Ranking:

{ranked}

Generate:

1. Executive Summary

2. Most Likely Causes

3. Confidence Level

4. Recommended Actions

Keep report concise.
"""

In [118]:
print(callGPT(prompt).text)

**1. Executive Summary**

The IAM investigation reveals significant underperformance, with SLA compliance at a concerning 72%. This is primarily driven by high approval latency (12 units) and a critical percentage of inactive approvers (45%). While a 5% campaign size growth adds some strain, the core issues stem from process inefficiencies exacerbated by a lack of active participation in approval workflows.

**2. Most Likely Causes**

*   **Inactive Approvers (45%):** A substantial portion of assigned approvers are not acting on requests, directly causing bottlenecks and approval delays.
*   **Process Inefficiency / Workflow Bottleneck:** Evidenced by the low SLA compliance and significant approval/provisioning latencies, indicating a suboptimal or blocked flow of requests.
*   **Campaign Growth (5%):** While moderate, consistent growth on an already inefficient system with inactive approvers further compounds delays and reduces overall compliance.

**3. Confidence Level**

**Moderate 